# Execução da camada Silver — ENEM por UF

Este notebook executa o processamento da camada **Silver** para as edições de **1998 a 2025**, filtrando e estruturando os dados para a Unidade Federativa definida em `UF_EXECUCAO`

### Regras de Operação:
- **Parametrização:** Para processar outro estado, altere apenas a variável `UF_EXECUCAO` na **Seção 1**
- **Isolamento:** Cada UF é gravada em seu próprio diretório particionado (`data/silver/uf=XX/`), sem misturar dados entre estados;
- **Idempotência:** Arquivos Parquet já existentes e validados para a UF são reutilizados automaticamente;
- **Estrutura 2024–2025:** Para as edições mais recentes, as bases de `participantes` e `resultados` são mantidas separadas;
- **Fail-fast:** A execução é interrompida imediatamente caso ocorra qualquer erro de validação ou leitura;

> **Nota:** Este fluxo não modifica as camadas **Bronze** nem **Gold**


## 1. Configuração e Importação de Módulos

Carrega as dependências do pipeline, normaliza a sigla da UF selecionada (`UF_EXECUCAO`) e define o diretório de destino na camada Silver

In [5]:
from _setup import *
from enem_pipeline.config import (
    PROJECT_ROOT, ANO_INICIAL, ANO_FINAL, SILVER_DIR,
    normalizar_uf, obter_diretorio_uf)

from enem_pipeline.silver import processar_ano_silver
import pandas as pd

UF_EXECUCAO = "RR"
UF_EXECUCAO = normalizar_uf(UF_EXECUCAO)
DIRETORIO_SILVER_UF = obter_diretorio_uf(SILVER_DIR, UF_EXECUCAO)

print("Raiz do projeto:", PROJECT_ROOT)
print("UF selecionada:", UF_EXECUCAO)
print("Período:", f"{ANO_INICIAL}–{ANO_FINAL}")
print("Destino Silver:", DIRETORIO_SILVER_UF)

Raiz do projeto: /home/akel/PycharmProjects/Projeto_ENEM
UF selecionada: RR
Período: 1998–2025
Destino Silver: /home/akel/PycharmProjects/Projeto_ENEM/data/silver/uf=RR


## 2. Conferência dos Parâmetros de Execução

Valida o intervalo de anos (1998–2025), verifica a existência do módulo `silver.py` e calcula o total de fontes esperadas (30 arquivos, considerando a divisão de bases em 2024 e 2025)

In [2]:
assert ANO_INICIAL == 1998
assert ANO_FINAL == 2025
assert len(UF_EXECUCAO) == 2
assert (PROJECT_ROOT / "src" / "enem_pipeline" / "silver.py").exists()

ANOS = list(range(ANO_INICIAL, ANO_FINAL + 1))
print(f"UF: {UF_EXECUCAO}")
print(f"Anos a percorrer: {len(ANOS)}")
print(f"Fontes Silver esperadas: {len(ANOS) + 2} (2024 e 2025 têm duas bases cada)")

UF: RR
Anos a percorrer: 28
Fontes Silver esperadas: 30 (2024 e 2025 têm duas bases cada)


## 3. Processamento Bronze/Staging → Silver

Executa a rotina `processar_ano_silver` para toda a série histórica da UF definida. Arquivos Parquet íntegros já presentes no destino são reaproveitados sem reprocessamento

In [3]:
resultados_silver = []

for posicao, ano in enumerate(ANOS, start=1):
    print(f"\n{'=' * 64}")
    print(f"[{posicao}/{len(ANOS)}] Processando edição {ano} — UF {UF_EXECUCAO}")
    print("=" * 64)

    try:
        resultados_ano = processar_ano_silver(ano, uf=UF_EXECUCAO)
        resultados_silver.extend(resultados_ano)

        for resultado in resultados_ano:
            origem = resultado.get("arquivo_origem", "fonte")
            status = resultado.get("status", "sem_status")
            destino = resultado.get("arquivo_saida", "")
            registros = resultado.get("total_registros")
            colunas = resultado.get("colunas_gravadas")

            print(f"Fonte: {origem}")
            print(f"Status: {status}")
            if destino:
                print(f"Saída: {destino}")
            if registros is not None:
                print(f"Registros: {registros:,} | Colunas: {colunas}")
            if resultado.get("validado") is False:
                raise RuntimeError(f"Validação não aprovada para {origem}, ano {ano}.")

    except Exception as erro:
        print(f"FALHA no ano {ano}: {type(erro).__name__}: {erro}")
        print("Execução interrompida; corrija a falha antes de continuar.")
        raise

print(f"\nExecução Silver concluída para todos os anos — UF {UF_EXECUCAO}.")


[1/28] Processando edição 1998 — UF RR
Parquet existente validado: /home/akel/PycharmProjects/Projeto_ENEM/data/silver/uf=RR/ano=1998/enem_1998.parquet
Fonte: MICRODADOS_ENEM_1998.csv
Status: ja_existente_validado
Saída: /home/akel/PycharmProjects/Projeto_ENEM/data/silver/uf=RR/ano=1998/enem_1998.parquet
Registros: 967 | Colunas: 27

[2/28] Processando edição 1999 — UF RR
Parquet existente validado: /home/akel/PycharmProjects/Projeto_ENEM/data/silver/uf=RR/ano=1999/enem_1999.parquet
Fonte: MICRODADOS_ENEM_1999.csv
Status: ja_existente_validado
Saída: /home/akel/PycharmProjects/Projeto_ENEM/data/silver/uf=RR/ano=1999/enem_1999.parquet
Registros: 1,014 | Colunas: 28

[3/28] Processando edição 2000 — UF RR
Parquet existente validado: /home/akel/PycharmProjects/Projeto_ENEM/data/silver/uf=RR/ano=2000/enem_2000.parquet
Fonte: MICRODADOS_ENEM_2000.csv
Status: ja_existente_validado
Saída: /home/akel/PycharmProjects/Projeto_ENEM/data/silver/uf=RR/ano=2000/enem_2000.parquet
Registros: 317 | Co

## 4. Resumo e Auditoria da Execução

Consolida os metadados retornados pelo processamento em um DataFrame para verificar a contagem de registros, colunas gravadas e o status de validação de cada fonte

In [6]:
df_resultados_silver = pd.DataFrame(resultados_silver)

if df_resultados_silver.empty:
    print("A rotina não retornou resultados.")
else:
    colunas_resumo = [
        coluna for coluna in [
            "ano", "arquivo_origem", "arquivo_saida", "total_registros",
            "colunas_gravadas", "validado", "status", "csv_removido"
        ]
        if coluna in df_resultados_silver.columns
    ]
    display(df_resultados_silver[colunas_resumo])
    print("Fontes retornadas:", len(df_resultados_silver))
    if "validado" in df_resultados_silver.columns:
        print("Fontes validadas:", int(df_resultados_silver["validado"].fillna(False).sum()))
    if "status" in df_resultados_silver.columns:
        display(df_resultados_silver["status"].value_counts(dropna=False).rename_axis("status").to_frame("quantidade"))

,ano,arquivo_origem,arquivo_saida,total_registros,colunas_gravadas,validado,status,csv_removido
0,1998,MICRODADOS_ENEM_1998.csv,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,967,27,True,ja_existente_validado,False
1,1999,MICRODADOS_ENEM_1999.csv,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,1014,28,True,ja_existente_validado,False
2,2000,MICRODADOS_ENEM_2000.csv,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,317,28,True,ja_existente_validado,False
3,2001,MICRODADOS_ENEM_2001.csv,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,2606,35,True,ja_existente_validado,False
4,2002,MICRODADOS_ENEM_2002.csv,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,5497,34,True,ja_existente_validado,False
5,2003,MICRODADOS_ENEM_2003.csv,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,6316,35,True,ja_existente_validado,False
6,2004,MICRODADOS_ENEM_2004.csv,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,4793,35,True,ja_existente_validado,False
7,2005,MICRODADOS_ENEM_2005.csv,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,5986,35,True,ja_existente_validado,False
8,2006,MICRODADOS_ENEM_2006.csv,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,2810,36,True,ja_existente_validado,False
9,2007,MICRODADOS_ENEM_2007.csv,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,4207,36,True,ja_existente_validado,False


Fontes retornadas: 30
Fontes validadas: 30


,quantidade
status,
ja_existente_validado,30


## 5. Critérios de Conclusão

A execução desta etapa é considerada bem-sucedida quando:
1. O laço de processamento encerra sem lançar exceções;
2. O resumo final contabiliza exatamente as 30 fontes esperada (26 edições de 1998–2023 e 4 bases de 2024–2025);
3. Todos os arquivos retornam com o indicador `validado = True`;
Os resultados ficam armazenados em `data/silver/uf={UF_EXECUCAO}/`. Para processar novas UFs, basta atualizar `UF_EXECUCAO` e reexecutar o notebook.